# Deep Hedging AI
### Example showing TensorBoard

Hans Buehler, January 2023


In [ ]:
""" Slighly annoying: by default the SageMaker Python import directory does not include our git directory """
!pip -q install cdxbasics "tensorflow>=2.10" "tensorflow_probability==0.14"
import os
p = os.getcwd()
dhn = "/deephedging/"
i = p.find(dhn)
if i!=-1:
    p = p[:i]
    import sys
    sys.path.append(p)
    print("SageMaker: added python path %s" % p)

In [ ]:
%matplotlib inline
print("Deep Hedging AI says hello ... ", end='')
from cdxbasics.config import Config
from cdxbasics.subdir import SubDir
from deephedging.trainer import train
from deephedging.gym import VanillaDeepHedgingGym
from deephedging.world import SimpleWorld_Spot_ATM

from IPython.display import display, Markdown

# see print of the config below for numerous options
config = Config()
# world
config.world.samples = 1000
config.world.steps = 10
config.world.black_scholes = True
# gym
config.gym.objective.utility = "cvar"
config.gym.objective.lmbda = 1.
config.gym.agent.network.depth = 3
config.gym.agent.network.depth = 3
config.gym.agent.network.activation = "softplus"
# trainer
config.trainer.train.batch_size = None
config.trainer.train.epochs = 10
config.trainer.visual.epoch_refresh = 2
config.trainer.visual.confidence_pcnt_lo = 0.25
config.trainer.visual.confidence_pcnt_hi = 0.75

# tensor board
# ------------
config.trainer.caching.mode = "off"  # caching makes no sense here
board_dir = SubDir("~/.tensorboard").path
config.trainer.train.tensor_board.log_dir = board_dir
config.trainer.train.tensor_board.hist_freq = 1
config.trainer.train.tensor_board.profile_batch  = '1,10'

# create world
world  = SimpleWorld_Spot_ATM( config.world )
val_world  = world.clone(samples=config.world("samples"))

# create training environment
gym = VanillaDeepHedgingGym( config.gym )

# create training environment
train( gym=gym, world=world, val_world=val_world, config=config.trainer )
config.done()



### TensorBoard

In [ ]:
# Load the TensorBoard notebook extension.
%reload_ext tensorboard
# Launch TensorBoard and navigate to the Profile tab to view performance profile

%tensorboard --logdir=board_dir



In [ ]:
#!kill 41456